## Python Practice

In [1]:
# Practice with control flows - if-elif-else

x = 10
if x > 0:
    print("Positive")
elif x < 0:
    print("Negative")
else:
    print("Zero")

Positive


In [2]:
# for loops
fruits = ["apple", "banana", "cherry"]
for fruit in fruits:
    print(fruit)

apple
banana
cherry


In [3]:
# while loop
count = 0
while count < 5:
    print(count)
    count += 1

0
1
2
3
4


In [4]:
# loop control
for i in range(10):
    if i ==5:
        break
    print(i)

0
1
2
3
4


In [5]:
for i in range(5):
    if i == 2:
        continue
    print(i)

0
1
3
4


## Emoji Visualizations
### [EDS 217 Cheatsheet](https://eds-217-essential-python.github.io/course-materials/cheatsheets/emoji_visualizations.html)

In [1]:
# Load in libraries
import matplotlib.pyplot as plt
import requests
from PIL import Image
import io
import numpy as np
import matplotlib.offsetbox as ob


In [ ]:
# Make a function 
def get_emoji_image(emoji, transparent_bg=True):
    """
    Download an emoji image from an online source and return as numpy array.

    Parameters:
    - emoji: str - A single emoji character
    - transparent bg: bool - If true, makes the background transparent

    Returns:
    - numpy array with shape (72, 72, 4) for RGBA or (72, 72, 3) for RGB
    """
    # Convert the emoji to Unicode codepoints
    # Handle multi-character emojis (e.g. emoji + variation selector)
    codepoints = []
    for char in emoji:
        codepoints.append(hex(ord(char))[2:].lower())

    # Join with underscores for multi-character emohis
    codepoint = "_".join(codepoints)

    # Try multiple emoji sources
    # Look for google emojis first 
    urls = [
        f"https://github.com/googlefonts/noto-emoji/raw/main/png/72/emoji_u{codepoint}.png",
        f"https://raw.githubusercontent.com/twitter/twemoji/master/assets/72x72/{codepoint}.png"
    ]

    # If multi-character, also try just the first character (base emoji)
    if len(codepoints) > 1:
        base_codepoint = codepoints[0]
        urls.extend([
            f"https://github.com/googlefonts/noto-emoji/raw/main/png/72/emoji_u{base_codepoint}.png",
            f"https://raw.githubusercontent.com/twitter/twemoji/master/assets/72x72/{base_codepoint}.png"
        ])

    for url in urls:
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                img = Image.open(io.BytesIO(response.content))

                # Keep transparency if it exists and requested
                if transparent_bg and img.mode in ('RGBA', 'LA'):
                    return np.array(img) # Keep alpha channel
                elif transparent_bg:
                    # Convert to RGBA and make white pixels transparent
                    img = img.convert("RGBA")
                    data = np.array(img)

                    # Make white/light pixels transparent
                    light_threshold = 240
                    mask = (data[:, :, 0] > light_threshold) & \
                            (data[:, :, 1] > light_threshold) & \
                            (data[:, :, 2] > light_threshold)
                    data[mask, 3] = 0 # Set alpha to 0 (transparent)
                    
                    return data
                
                else:
                    # No transparency requested
                    return np.array(img.convert("RGB"))
                                    
        except:
                continue
        

        # Fallback: create a colored square
        if transparent_bg:
            img = Image.new('RGBA', (72, 72), color=(200, 200, 200, 255))
        else:
            img = Image.new('RGB', (72, 72), color=(200, 200, 200))
        return np.array(img)
    
